# 비지터(Visitor) 패턴
데이터 구조(객체)와 그 데이터 위에서 돌아가는 로직(알고리즘)을 분리하는 패턴
새로운 기능을 추가하고 싶을 때, 원래 있던 객체의 코드를 건드리지 않고도 '방문자(Visitor)'만 새로 만들어서 추가하면 된다.

## 장점
- **객체 수정 최소화**: 기존 클래스의 코드를 변경하지 않고도 새로운 연산을 추가할 수 있습니다. (개방-폐쇄 원칙 준수)
- **데이터와 로직의 분리**: 데이터 구조는 순수하게 데이터만 담고, 복잡한 계산은 방문자가 전담합니다.
- **관련 코드 집중**: 흩어져 있던 비슷한 로직들을 하나의 Visitor 클래스에 모을 수 있습니다.

## 주요 구성 요소
- Element (요소 인터페이스): 방문자를 받아들이는 accept() 메서드를 정의합니다.
- Concrete Element (실제 요소): 실제로 데이터가 들어있는 객체입니다.
- Visitor (방문자 인터페이스): 각 객체를 방문했을 때 수행할 작업들을 정의합니다.
- Concrete Visitor (실제 방문자): 구체적으로 어떤 로직을 실행할지 구현합니다.

In [1]:
// 1. Element: 방문자를 받아들일 수 있는 인터페이스
interface CartItem {
    fun accept(visitor: ShoppingVisitor): Int
}

// 2. Concrete Elements: 실제 데이터 객체들
class Fruit(val price: Int, val weight: Int) : CartItem {
    override fun accept(visitor: ShoppingVisitor) = visitor.visit(this)
}

class Electronics(val price: Int) : CartItem {
    override fun accept(visitor: ShoppingVisitor) = visitor.visit(this)
}

// 3. Visitor: 방문자 인터페이스
interface ShoppingVisitor {
    fun visit(fruit: Fruit): Int
    fun visit(electronics: Electronics): Int
}

// 4. Concrete Visitor: 실제 로직 (가격 계산기)
class PriceCalculator : ShoppingVisitor {
    override fun visit(fruit: Fruit): Int = fruit.price * fruit.weight
    override fun visit(electronics: Electronics): Int = (electronics.price * 1.1).toInt() // 부가세 포함
}

val items = listOf(Fruit(2000, 3), Electronics(50000))
val calculator = PriceCalculator()

val total = items.sumOf { it.accept(calculator) }
println("총 가격: $total")

총 가격: 61000


## 단점
- **구조 변경의 취약성**: 새로운 종류의 객체(예: Vegetable)가 추가되면 모든 Visitor의 코드를 다 수정해야 합니다. (객체 구조가 고정적일 때 유리함)
- **캡슐화 약화**: Visitor가 일을 하려면 객체의 내부 데이터를 다 꺼내봐야 하므로, 객체의 속성들을 외부에 노출하게 될 가능성이 큽니다.

---
## 트래버서(Traverser)
비지터 패턴이 "무엇을 할지"에 집중한다면, **트래비서(Traverser)** 는 그 작업을 "어떤 순서로, 어떻게 전달할지"를 전담

### 결합의 장점 (요약)
- **교체 가능성**: 방문자(로직)는 그대로 두고, 트래비서만 바꾸면 순회 순서(예: 이름순 vs 날짜순)를 쉽게 바꿀 수 있습니다.
- **코드 깔끔함**:
    - Element: 데이터만 가짐.
    - Visitor: 로직만 가짐.
    - Traverser: 길 찾기만 함.

In [2]:
// 1. Element: 방문할 대상 (직원, 부서)
interface Node {
    fun accept(visitor: Visitor)
}

class Employee(val name: String, val salary: Int) : Node {
    override fun accept(visitor: Visitor) = visitor.visit(this)
}

class Department(val name: String) : Node {
    val children = mutableListOf<Node>()
    override fun accept(visitor: Visitor) = visitor.visit(this)
}

// 2. Visitor: 수행할 작업 (급여 합계 계산)
interface Visitor {
    fun visit(employee: Employee)
    fun visit(department: Department)
}

class SalarySumVisitor : Visitor {
    var total = 0
    override fun visit(employee: Employee) {
        total += employee.salary
    }

    override fun visit(department: Department) { /* 부서 자체는 급여가 없음 */
    }
}

// 3. Traverser: 길 안내자 (가장 중요한 부분!)
class OrganizationTraverser(val root: Node) {
    fun walk(visitor: Visitor) {
        recursiveWalk(root, visitor)
    }

    private fun recursiveWalk(node: Node, visitor: Visitor) {
        // [핵심] 방문자에게 현재 노드를 처리하게 함
        node.accept(visitor)

        // [핵심] 만약 부서(구조)라면 자식들을 순회하도록 안내함
        if (node is Department) {
            node.children.forEach { child ->
                recursiveWalk(child, visitor)
            }
        }
    }
}

// 4. 실행
// 구조 생성
val ceo = Employee("CEO", 10000)
val devDept = Department("개발팀")
devDept.children.add(Employee("개발자A", 5000))
devDept.children.add(Employee("개발자B", 6000))

val root = Department("회사")
root.children.add(ceo)
root.children.add(devDept)

// 로직(Visitor)과 가이드(Traverser) 준비
val salaryVisitor = SalarySumVisitor()
val traverser = OrganizationTraverser(root)

// 실행: 트래비서가 비지터를 데리고 이동함
traverser.walk(salaryVisitor)

println("총 급여 합계: ${salaryVisitor.total}")

총 급여 합계: 21000
